In [ ]:
import os
import numpy as np
import sys
import joblib
import pandas as pd

# Get the directory of the script
script_dir = os.getcwd()

# Get the parent directory of the script
parent_dir = os.path.dirname(script_dir)

# Add the parent directory to sys.path
sys.path.append(parent_dir)

from core.autoencoders import AE, train_ae
from core.dataset import TIFFDataset
# from utils.feature_analysis import UMAP_train, patch_csv_to_AE_latent, patch_2_normed_tensor,data_to_latents, kmeans_latents,latent_to_umap,UMAP_train,DBSCAN_cluster,kmeans_cluster,dataloader_model_latents,add_features_to_latent
# from utils.plotting_utils import umap_2Dplot, cluster_2Dplot

import torch
# Load Data
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
root_dir = '/mnt/d/lding/CLS_GitHub/fa_patch_AE_clustering/results/pax_wholecell_correctloader_ch1_ps32_latdim_8_20251203_2323'

ae_path = os.path.join(root_dir,'ae_model_ep9999.pt')
ae = torch.load(ae_path, map_location=device, weights_only=False)
ae = ae.to(device)
ae.eval()

In [ ]:
csv_folder = '/mnt/d/lding/FA/analysis_results/FA_ML_Annabel_20250217/031125/ctrl_ch1_major/ctrl_ch1_patches_gridonly_wholecell_pslocation00/plot_patches32_40p_20250919_0948'
csv_filename = 'data_prep_record_49_t49.csv'

pad_size = 64
import pandas as pd

all_image_csv = pd.read_csv(os.path.join(csv_folder,csv_filename))

unique_vals = all_image_csv["filename"].unique()
latent_array =  []
index_count=-1

for filename in unique_vals:
    this_file_csv = all_image_csv[all_image_csv["filename"]==filename]
    # print(csv_filename)
    raw_whole_image = np.zeros([1024,1024])
    recon_whole_image = np.zeros([1024,1024])    
    # raw_whole_image_RGB = np.zeros([3,1024,1024])
    # recon_whole_image_RGB = np.zeros([3,1024,1024])    

    for index, row in this_file_csv.iterrows():       
        index_count = index_count+1 
        x_corner1 = int(row["x_corner1"])
        x_corner3 = int(row["x_corner3"])
        y_corner1 = int(row["y_corner1"])
        y_corner3 = int(row["y_corner3"])
        patch_name = row['crop_img_filename']

                
        raw_patch = tiff.imread(os.path.join(row["movie_partitioned_data_dir"],row["crop_img_filename"] ))
        normed_raw_patch = raw_patch.copy() * 240 
        normed_raw_patch[normed_raw_patch > 254] = 254
        normed_raw_patch = normed_raw_patch/255
        
        tensor_patch = torch.from_numpy(normed_raw_patch)
        tensor_patch = tensor_patch.unsqueeze(0).unsqueeze(0)
        tensor_patch = tensor_patch.to(device)
        ae = ae.to(device)
        with torch.no_grad():
            recon_image, latent = ae(tensor_patch)
        latent_array.append(latent)        

        recon_patch_filename = 'nc'+str(int(kmeans_labels[index_count])).zfill(3) +"_recon_patch_"+patch_name+'.tif'
        output_dir_cindex = os.path.join(recon_patch_dir,'c'+str(int(kmeans_labels[index_count])).zfill(3))
        os.makedirs(output_dir_cindex, exist_ok=True)
        tiff.imwrite(
            os.path.join(output_dir_cindex,recon_patch_filename),
            recon_image.squeeze().cpu().detach().numpy().astype(np.float32),
            imagej=True,              # Write ImageJ metadata block
            metadata={'axes': 'YX'}   # Or 'TYX', 'ZYX', etc. depending on shape
        )    
        
        raw_patch_img_filename = 'c'+str(int(kmeans_labels[index_count])).zfill(3) +"_raw_patch_"+patch_name+'.tif'
        output_dir_cindex = os.path.join(raw_patch_dir,'c'+str(int(kmeans_labels[index_count])).zfill(3))
        os.makedirs(output_dir_cindex, exist_ok=True)
        tiff.imwrite(
            os.path.join(output_dir_cindex,raw_patch_img_filename),
            normed_raw_patch.astype(np.float32),
            imagej=True,              # Write ImageJ metadata block
            metadata={'axes': 'YX'}   # Or 'TYX', 'ZYX', etc. depending on shape
        )
        raw_whole_image[y_corner1-pad_size:y_corner3-pad_size,x_corner1-pad_size:x_corner3-pad_size] = normed_raw_patch
        recon_whole_image[y_corner1-pad_size:y_corner3-pad_size,x_corner1-pad_size:x_corner3-pad_size] = recon_image.squeeze().cpu().detach()
        recon_whole_image_RGB[0,y_corner1-pad_size:y_corner3-pad_size,x_corner1-pad_size:x_corner3-pad_size] = recon_image.squeeze().cpu().detach()
        recon_whole_image_RGB[1,y_corner1-pad_size:y_corner3-pad_size,x_corner1-pad_size:x_corner3-pad_size] = recon_image.squeeze().cpu().detach()
        recon_whole_image_RGB[2,y_corner1-pad_size:y_corner3-pad_size,x_corner1-pad_size:x_corner3-pad_size] = recon_image.squeeze().cpu().detach()
        if(kmeans_labels[index_count]==0):
            recon_whole_image_RGB[0,y_corner1-pad_size:y_corner3-pad_size,x_corner1-pad_size:x_corner3-pad_size] =1
        if(kmeans_labels[index_count]==1):
            recon_whole_image_RGB[1,y_corner1-pad_size:y_corner3-pad_size,x_corner1-pad_size:x_corner3-pad_size] =1
        if(kmeans_labels[index_count]==2):
            recon_whole_image_RGB[2,y_corner1-pad_size:y_corner3-pad_size,x_corner1-pad_size:x_corner3-pad_size] =1



    fig, ax = plt.subplots(1,2, figsize=(8,4))
    ax[0].imshow(raw_whole_image,cmap=plt.cm.gray)
    ax[1].imshow(recon_whole_image,cmap=plt.cm.gray)

    recon_img_filename = "recon_"+filename+'.tif'
    
    tiff.imwrite(
        os.path.join(recon_dir,recon_img_filename),
        recon_whole_image.astype(np.float32),#.swapaxes(0,2),
        imagej=True,              # Write ImageJ metadata block
        metadata={'axes': 'YX'}   # Or 'TYX', 'ZYX', etc. depending on shape
    )    
    
    raw_grid_img_filename = "raw_grid_"+filename+'.tif'
    
    tiff.imwrite(
        os.path.join(raw_grid_dir,raw_grid_img_filename),
        raw_whole_image.astype(np.float32),
        imagej=True,              # Write ImageJ metadata block
        metadata={'axes': 'YX'}   # Or 'TYX', 'ZYX', etc. depending on shape
    )

    break